In [1]:
import sys
import logging
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

logging.basicConfig(level=logging.CRITICAL, force=True, stream=sys.stdout)
# logging.getLogger('torch_numopt.numerical_optimizer').setLevel(logging.INFO)
# logging.
n_epoch = 1000
# n_epoch = 10

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(model.parameters(), line_search_cond="goldstein", line_search_method="bisect")

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=n_epoch, max_patience=50)

epoch:  0, loss: 0.04247058928012848
epoch:  1, loss: 0.03778683394193649
epoch:  2, loss: 0.03701745346188545
epoch:  3, loss: 0.03695115074515343
epoch:  4, loss: 0.03687664493918419
epoch:  5, loss: 0.03686898574233055
epoch:  6, loss: 0.03682824969291687
epoch:  7, loss: 0.03680938109755516
epoch:  8, loss: 0.036783814430236816
epoch:  9, loss: 0.03676155209541321
epoch:  10, loss: 0.036748215556144714
epoch:  11, loss: 0.03671839088201523
epoch:  12, loss: 0.036698658019304276
epoch:  13, loss: 0.03669394552707672
epoch:  14, loss: 0.03665248677134514
epoch:  15, loss: 0.03663692995905876
epoch:  16, loss: 0.03660840913653374
epoch:  17, loss: 0.036587223410606384
epoch:  18, loss: 0.036579761654138565
epoch:  19, loss: 0.03654111176729202
epoch:  20, loss: 0.036523327231407166
epoch:  21, loss: 0.036496058106422424
epoch:  22, loss: 0.0364723727107048
epoch:  23, loss: 0.03646301105618477
epoch:  24, loss: 0.0364215113222599
epoch:  25, loss: 0.03640104830265045
epoch:  26, loss:

In [6]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7309567928314209
Test metrics:  R2 = 0.6671619415283203


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(model.parameters(), line_search_cond="armijo", line_search_method="backtrack")

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=n_epoch, max_patience=50)

epoch:  0, loss: 0.2011832445859909
epoch:  1, loss: 0.1251947432756424
epoch:  2, loss: 0.08345135301351547
epoch:  3, loss: 0.06095382198691368
epoch:  4, loss: 0.04909515380859375
epoch:  5, loss: 0.042974378913640976
epoch:  6, loss: 0.03987007588148117
epoch:  7, loss: 0.03831735625863075
epoch:  8, loss: 0.03754844143986702
epoch:  9, loss: 0.037170372903347015
epoch:  10, loss: 0.03698454424738884
epoch:  11, loss: 0.036892011761665344
epoch:  12, loss: 0.03684527426958084
epoch:  13, loss: 0.036820657551288605
epoch:  14, loss: 0.0368066243827343
epoch:  15, loss: 0.03679295629262924
epoch:  16, loss: 0.036768391728401184
epoch:  17, loss: 0.03675440326333046
epoch:  18, loss: 0.03673979267477989
epoch:  19, loss: 0.036715488880872726
epoch:  20, loss: 0.0367015078663826
epoch:  21, loss: 0.0366864837706089
epoch:  22, loss: 0.036661699414253235
epoch:  23, loss: 0.03664756566286087
epoch:  24, loss: 0.03663112223148346
epoch:  25, loss: 0.03660644590854645
epoch:  26, loss: 0.

In [8]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7135740518569946
Test metrics:  R2 = 0.6690847277641296


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(model.parameters(), line_search_cond="goldstein", line_search_method="backtrack")

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=n_epoch, max_patience=50)

epoch:  0, loss: 0.20534737408161163
epoch:  1, loss: 0.1171930655837059
epoch:  2, loss: 0.07470883429050446
epoch:  3, loss: 0.05457383021712303
epoch:  4, loss: 0.04540427029132843
epoch:  5, loss: 0.04099421948194504
epoch:  6, loss: 0.03882504254579544
epoch:  7, loss: 0.03774544969201088
epoch:  8, loss: 0.037201523780822754
epoch:  9, loss: 0.03692256659269333
epoch:  10, loss: 0.03677552565932274
epoch:  11, loss: 0.0366922952234745
epoch:  12, loss: 0.03663933277130127
epoch:  13, loss: 0.03649714216589928
epoch:  14, loss: 0.03642075136303902
epoch:  15, loss: 0.036376532167196274
epoch:  16, loss: 0.036350201815366745
epoch:  17, loss: 0.036259740591049194
epoch:  18, loss: 0.03621451184153557
epoch:  19, loss: 0.036187972873449326
epoch:  20, loss: 0.036129292100667953
epoch:  21, loss: 0.036076620221138
epoch:  22, loss: 0.03604951128363609
epoch:  23, loss: 0.035994190722703934
epoch:  24, loss: 0.03594455495476723
epoch:  25, loss: 0.03591766580939293
epoch:  26, loss: 0

In [10]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7531979084014893
Test metrics:  R2 = 0.6977202892303467


In [11]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(model.parameters(), line_search_cond="wolfe", line_search_method="interpolate")

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=n_epoch, max_patience=50)

epoch:  0, loss: 0.03925153985619545
epoch:  1, loss: 0.037120018154382706
epoch:  2, loss: 0.03708973899483681
epoch:  3, loss: 0.0370635986328125
epoch:  4, loss: 0.03704361245036125
epoch:  5, loss: 0.03703182190656662
epoch:  6, loss: 0.03700803592801094
epoch:  7, loss: 0.036990489810705185
epoch:  8, loss: 0.03697553649544716
epoch:  9, loss: 0.03696943446993828
epoch:  10, loss: 0.0369456522166729
epoch:  11, loss: 0.03693239763379097
epoch:  12, loss: 0.03692145645618439
epoch:  13, loss: 0.036914680153131485
epoch:  14, loss: 0.036901362240314484
epoch:  15, loss: 0.03689117729663849
epoch:  16, loss: 0.036883361637592316
epoch:  17, loss: 0.03688039630651474
epoch:  18, loss: 0.03686593100428581
epoch:  19, loss: 0.036857131868600845
epoch:  20, loss: 0.03684936836361885
epoch:  21, loss: 0.03684398904442787
epoch:  22, loss: 0.03683420270681381
epoch:  23, loss: 0.03682591766119003
epoch:  24, loss: 0.03681916743516922
epoch:  25, loss: 0.03681626543402672
epoch:  26, loss: 

In [12]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7403548955917358
Test metrics:  R2 = 0.6960527896881104


In [13]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(model.parameters(), line_search_cond="strong-wolfe", line_search_method="interpolate")

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=n_epoch, max_patience=50)

epoch:  0, loss: 0.04587947577238083
epoch:  1, loss: 0.035596318542957306
epoch:  2, loss: 0.03543499484658241
epoch:  3, loss: 0.035336144268512726
epoch:  4, loss: 0.035023607313632965
epoch:  5, loss: 0.03475421294569969
epoch:  6, loss: 0.03459342196583748
epoch:  7, loss: 0.03455141931772232
epoch:  8, loss: 0.034162186086177826
epoch:  9, loss: 0.03387446701526642
epoch:  10, loss: 0.03377088904380798
epoch:  11, loss: 0.03343461826443672
epoch:  12, loss: 0.03315598517656326
epoch:  13, loss: 0.032969821244478226
epoch:  14, loss: 0.03283967822790146
epoch:  15, loss: 0.032474398612976074
epoch:  16, loss: 0.032160766422748566
epoch:  17, loss: 0.03194175660610199
epoch:  18, loss: 0.031774140894412994
epoch:  19, loss: 0.0313679575920105
epoch:  20, loss: 0.031007643789052963
epoch:  21, loss: 0.030749324709177017
epoch:  22, loss: 0.03053828701376915
epoch:  23, loss: 0.03008146770298481
epoch:  24, loss: 0.0296742245554924
epoch:  25, loss: 0.029357433319091797
epoch:  26, l

In [14]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7261694073677063
Test metrics:  R2 = 0.6619239449501038
